# Micrograd From Scratch

## Author: RAHUL KP KURUP

This notebook implemtes a minimal automatic differentiation engine and builds a neural network (MLP) from scratch.

# Features
- Scalar-based autograd engine
- Backpropagation using topological sort
- Activation Function: tanh, ReLu, exp
- Neural network: Neuron -> Layer -> MLP
- Visualization support (Graphviz)

# 1. Core Autograd Engine

## "__repr__" function

In [1]:
import math
class sample:
    def __init__(self, data):
        self.data = data
        self.grad = 0

    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"

v = sample(5)
print(v)

Value(data=5, grad=0)


## "__add__" function

In [ ]:
class Value:
    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self.grad = 0
        self._children = _children   # computation graph
        self._op = _op               # operation
        self._backward = lambda: None  # default empty function

    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"

    def __add__(self, other):
        # Ensure both are Value objects
        other = other if isinstance(other, Value) else Value(other)
        
        # Create output node
        out = Value(self.data + other.data, (self, other), '+')

        # Define backward function (INSIDE)
        def _backward():
        # Gradient rule:
        # d/dx (x + x) = 2
        # More generally:
        # d/dself (self + other) = 1
        # d/dother (self + other) = 1
        # So each input receives the full upstream gradient (out.grad)
            self.grad += out.grad
            other.grad += out.grad

        # Attach backward function to output node
        out._backward = _backward

        return out
v1 = Value(10)
# v2 = Value(5)
v2 = 10
result = v1 + v2
print(result)
    

Value(data=20, grad=0)


# Gradient of Addition

We want to compute the derivative:

$$\frac{d}{dx}(x + x) = ?$$

## Step-by-step

Simplify the expression:

$$x + x = 2x$$

Now differentiate:

$$\frac{d}{dx}(2x) = 2$$

## Final Answer

$$\frac{d}{dx}(x + x) = 2$$

##  Intuition

- Each variable contributes a gradient of **1**
- So:
  - First x → 1
  - Second x → 1

 Total:

$$1 + 1 = 2$$

In [ ]:
# Simple numerical verification
import numpy as np

def f(x):
    return x + x

x = 5
h = 1e-5

# Numerical derivative 
grad = (f(x + h) - f(x - h)) / (2 * h)

print('Numerical Gradient:', grad)

Numerical Gradient: 1.9999999999242843


# Full Engine

In [ ]:
import math

class Value:

    def __init__(self, data, children=(), _op='', label=''):
        self.data = float(data)
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(children)
        self._op = _op
        self.label = label
    
    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"
    
    def __add__(self, other):
        other = other if isinstance (other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')

        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        
        out._backward = _backward
        return out

